# 03 Stress Tests

Thin notebook over the phase script(s) in `scripts/`. Cells are split at the `# ----` section markers. Run from the repo root (`jupyter lab`), or just run the scripts.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
%matplotlib inline

## scripts/20_phase2.py

In [ ]:
"""Phase 2: stress-test the lift number.

2a pull-forward / pantry loading, 2b promo-type comparison (Welch + BH), 2c penetration vs volume decomposition.
Outputs: data/processed/events_phase2.parquet, outputs/phase2_results.md, outputs/phase2_summary.json, figures.
"""
import json
import sys
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean, PROC  # noqa: E402
from src.panel import build_panel  # noqa: E402
from src import metrics as M  # noqa: E402
from src.bootstrap import summarise  # noqa: E402

FIG = ROOT / "outputs" / "figures"
OUT = ROOT / "outputs" / "phase2_results.md"
lines = []; P = lines.append
summary = {}

df = build_clean()
preds = pd.read_parquet(PROC / "predictions.parquet")
ev = pd.read_parquet(PROC / "events.parquet")
pan = build_panel(df, preds)

P("# Phase 2 results (generated by scripts/20_phase2.py)\n")

In [ ]:
# ------------------------------------------------------------------ 2a pull-forward
win = M.post_and_pre_windows(pan, ev)
ev = ev.merge(win, on="event_id", how="left")
ev["net_incr"] = ev["incr_A"] + ev["post_incr"]
ev["pullforward_ratio_event"] = 1 - ev["net_incr"] / ev["incr_A"]

P("## 2a. Pull-forward / pantry loading\n")
full = ev[ev.n_post_clean == M.POST_WEEKS]
P(f"- Post-window rule: weeks t1+1..t1+4 after the run ends, truncated at the first promoted or unobserved week. "
  f"{(ev.n_post_clean==4).mean():.0%} of events ({len(full):,}) have a full clean 4-week window; "
  f"{(ev.n_post_clean==0).mean():.0%} have none (another promotion started the very next week).")
pf = M.pullforward_summary(ev)
summary["pullforward"] = pf
P(f"- Headline (events with a full clean post window, n={pf['n_events']:,}):")
P(f"  - In-week (run) lift: **{pf['in_week_lift_pct']:.1%}** (95% CI {pf['in_week_lift_lo']:.1%} to {pf['in_week_lift_hi']:.1%})")
P(f"  - Net lift over run + 4 weeks: **{pf['net_lift_pct']:.1%}** (95% CI {pf['net_lift_lo']:.1%} to {pf['net_lift_hi']:.1%})")
P(f"  - Pull-forward ratio = 1 - net/in-week: **{pf['pullforward_ratio']:+.1%}** (95% CI {pf['pullforward_lo']:+.1%} to {pf['pullforward_hi']:+.1%})")
P(f"  - Mean incremental units per event: in-week {pf['in_week_incr_mean']:.1f}, post-window {pf['post_incr_mean']:+.1f}, net {pf['net_incr_mean']:.1f}")
sign = "overstates" if pf["pullforward_ratio"] > 0 else "UNDERSTATES"
P(f"- Reading: the naive in-week figure {sign} true incrementality by {abs(pf['pullforward_ratio']):.1%}. "
  "A positive ratio means post-promotion weeks dip below baseline (stock-up); a negative one means they stay above it (carry-over).")

# local-demand adjustment: the pre-window residual (actual - counterfactual in t0-4..t0-1) measures how much the
# baseline is locally off around a promotion (promotions are timed into locally strong weeks). Subtracting it from
# the post-window residual is a conservative check that a dip is not being masked.
pre_cols = [f"pre_{k}" for k in range(1, M.PRE_WEEKS + 1)]
ev["pre_resid_mean"] = ev[pre_cols].mean(axis=1)
ev["post_incr_adj"] = ev["post_incr"] - ev["n_post_clean"] * ev["pre_resid_mean"].fillna(0)
fa = ev[(ev.n_post_clean == M.POST_WEEKS) & ev.pre_resid_mean.notna()]
adj_ratio = 1 - (fa.incr_A + fa.post_incr_adj).sum() / fa.incr_A.sum()
rng = np.random.default_rng(1); idx = rng.integers(0, len(fa), size=(1000, len(fa)))
inw = fa.incr_A.to_numpy(); pa_ = fa.post_incr_adj.to_numpy()
adj_draw = 1 - (inw[idx] + pa_[idx]).sum(axis=1) / inw[idx].sum(axis=1)
summary["pullforward_adjusted"] = {"ratio": float(adj_ratio), "lo": float(np.percentile(adj_draw, 2.5)), "hi": float(np.percentile(adj_draw, 97.5)),
                                   "pre_resid_per_week": float(fa.pre_resid_mean.mean()), "post_resid_per_week": float(fa.post_incr.mean() / 4), "n": int(len(fa))}
P(f"- Local-demand check: in the 4 weeks BEFORE a promotion the baseline residual averages {fa.pre_resid_mean.mean():+.2f} units/week "
  f"(promotions land in locally strong weeks); in the 4 weeks after it averages {fa.post_incr.mean()/4:+.2f} units/week. "
  f"Subtracting the pre-residual from the post-window gives an adjusted pull-forward ratio of **{adj_ratio:+.1%}** "
  f"(95% CI {np.percentile(adj_draw, 2.5):+.1%} to {np.percentile(adj_draw, 97.5):+.1%}). Still no dip.")

# by category
cat_pf = M.compare_pullforward(ev, "category", sorted(ev.category.unique()))
summary["pullforward_by_category"] = cat_pf.to_dict(orient="records")
P("\n### Pull-forward by category\n")
P(cat_pf.to_markdown(index=False, floatfmt=".3f"))
# by promo type
pt_pf = M.compare_pullforward(ev, "promo_type", [t for t in ev.promo_type.value_counts().index if (full.promo_type == t).sum() >= 100])
P("\n### Pull-forward by promotion type\n")
P(pt_pf.to_markdown(index=False, floatfmt=".3f"))
# by discount depth
full = ev[ev.n_post_clean == M.POST_WEEKS].copy()
full["depth_bin"] = pd.cut(full.discount_depth, [-0.01, 0.05, 0.15, 0.25, 0.35, 1.0], labels=["<=5% (no cut)", "5-15%", "15-25%", "25-35%", ">35%"])
dd = M.compare_pullforward(full, "depth_bin", list(full.depth_bin.cat.categories))
P("\n### Pull-forward and lift by discount depth (checklist: deeper discounts should lift more)\n")
ld = summarise(ev, "incr_A", "baseline_units", by=["depth_bin"] if "depth_bin" in ev else None) if False else None
ev["depth_bin"] = pd.cut(ev.discount_depth, [-0.01, 0.05, 0.15, 0.25, 0.35, 1.0], labels=["<=5% (no cut)", "5-15%", "15-25%", "25-35%", ">35%"])
ld = summarise(ev, "incr_A", "baseline_units", by=["depth_bin"], n_boot=500)
ld = ld.merge(dd[["depth_bin", "pullforward_ratio", "lo", "hi"]], on="depth_bin", how="left").rename(columns={"lo": "pf_lo", "hi": "pf_hi"})
P(ld.to_markdown(index=False, floatfmt=".3f"))
summary["lift_by_depth"] = ld.to_dict(orient="records")

# event study
es = M.event_study(ev)
es.to_csv(ROOT / "outputs" / "phase2_event_study.csv", index=False)
es_cat = {c: M.event_study(ev[ev.category == c], n_boot=300) for c in sorted(ev.category.unique())}
pd.concat([d.assign(category=c) for c, d in es_cat.items()]).to_csv(ROOT / "outputs" / "phase2_event_study_by_category.csv", index=False)
single = ev[ev.run_length == 1]
es1 = M.event_study(single)

def _plot_es(d, ax, title, color="#1f6f8b"):
    x = d.rel_week.to_numpy(); y = d.mean_incr.to_numpy()
    ax.fill_between(x, d.lo, d.hi, alpha=.2, color=color, label="95% bootstrap CI (event level)")
    ax.plot(x, y, marker="o", color=color, label="mean incremental units / week")
    ax.axhline(0, color="k", lw=.8); ax.axvline(0, color="grey", ls=":", lw=.8)
    ax.set_xticks(x); ax.set_xticklabels([("promo" if v == 0 else f"{v:+d}") for v in x])
    ax.set_xlabel("week relative to promotion (0 = per-week average inside the run)")
    ax.set_ylabel("actual - counterfactual units"); ax.set_title(title); ax.grid(alpha=.3); ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
_plot_es(es, axes[0], f"All events (n={len(ev):,})")
_plot_es(es1, axes[1], f"Single-week promotions only (n={len(single):,})")
fig.tight_layout(); fig.savefig(FIG / "phase2_event_study.png", dpi=130); plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, (c, d) in zip(axes.ravel(), es_cat.items()):
    _plot_es(d, ax, c.title())
fig.tight_layout(); fig.savefig(FIG / "phase2_event_study_by_category.png", dpi=130); plt.close(fig)
P("\n### Event study (figures: outputs/figures/phase2_event_study.png, phase2_event_study_by_category.png)\n")
P(es.to_markdown(index=False, floatfmt=".2f"))
P("\nPost-week means by category (units/week, actual minus counterfactual):\n")
P(pd.DataFrame({c: d.set_index("rel_week").mean_incr for c, d in es_cat.items()}).T.to_markdown(floatfmt=".2f"))

In [ ]:
# ------------------------------------------------------------------ 2b promo types
P("\n## 2b. Which promotion types work\n")
ev["lift_A_pct_wk"] = ev["incr_A_per_week"] / (ev["baseline_units"] / ev["run_length"])
ts = summarise(ev, "incr_A", "baseline_units", by=["promo_type"], n_boot=1000).sort_values("lift_pct", ascending=False)
ts_wk = ev.groupby("promo_type").agg(incr_per_week=("incr_A_per_week", "mean"), mean_discount=("discount_depth", "mean"), mean_run_len=("run_length", "mean")).reset_index()
ts = ts.merge(ts_wk, on="promo_type")
summary["promo_types"] = ts.to_dict(orient="records")
P("Lift % = sum incremental / sum baseline over the run, bootstrap CI at event level. `incr_per_week` = mean incremental units per promoted week.\n")
P(ts.to_markdown(index=False, floatfmt=".3f"))
summ, pw = M.promo_type_tests(ev, value_col="lift_A_pct")
P("\n### Pairwise Welch t-tests on per-event lift %, Benjamini-Hochberg corrected\n")
P(pw.to_markdown(index=False, floatfmt=".4f"))
P(f"\n- {int(pw.significant_bh_05.sum())} of {len(pw)} pairwise differences survive BH at 5%.")
pw.to_csv(ROOT / "outputs" / "phase2_promo_type_tests.csv", index=False)
ts.to_csv(ROOT / "outputs" / "phase2_promo_type_lift.csv", index=False)
dvp = pw[((pw.type_a == "display_only") & (pw.type_b == "price_only")) | ((pw.type_a == "price_only") & (pw.type_b == "display_only"))]
if len(dvp):
    r = dvp.iloc[0]
    P(f"- Display alone vs price cut alone: mean per-event lift {r.mean_a:.1%} ({r.type_a}) vs {r.mean_b:.1%} ({r.type_b}), "
      f"BH-adjusted p = {r.p_bh:.4f}. Cost comparison is in Phase 4 (allocation), where cost per lever is a parameter.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ts.promo_type, ts.lift_pct * 100, yerr=[(ts.lift_pct - ts.lift_lo) * 100, (ts.lift_hi - ts.lift_pct) * 100], capsize=4, color="#1f6f8b")
ax.set_ylabel("incremental lift, % of baseline units"); ax.set_title("Lift by promotion type (95% event-level bootstrap CI)")
ax.tick_params(axis="x", rotation=30); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase2_lift_by_promo_type.png", dpi=130); plt.close(fig)

In [ ]:
# ------------------------------------------------------------------ 2c decomposition
P("\n## 2c. Where does the lift come from: new buyers or bigger baskets?\n")
ev = M.decomposition(ev)
ok = ev[ev.driver != "no_lift"]
P(f"- Identity: ln(units/baseline) = ln(hhs/baseline_hhs) + ln(units-per-hh / baseline). Both baselines are the Phase 1 "
  "gradient-boosting models (units and households fitted with the same design on the same clean weeks).")
P(f"- Events with meaningful lift (log lift > 0.10): {len(ok):,} of {len(ev):,}. Among them, the households component "
  f"explains on average **{ok.pen_share.clip(-1, 2).mean():.0%}** of the log lift (median {ok.pen_share.median():.0%}).")
P(f"- Classification: **{(ok.driver=='penetration').mean():.0%} penetration-driven**, {(ok.driver=='volume').mean():.0%} volume-driven.")
dc = ok.groupby("category").agg(events=("event_id", "size"), pen_share_mean=("pen_share", lambda s: s.clip(-1, 2).mean()),
                                 share_penetration_driven=("driver", lambda s: (s == "penetration").mean())).reset_index()
P("\nBy category:\n"); P(dc.to_markdown(index=False, floatfmt=".3f"))
dt = ok.groupby("promo_type").agg(events=("event_id", "size"), pen_share_mean=("pen_share", lambda s: s.clip(-1, 2).mean()),
                                   share_penetration_driven=("driver", lambda s: (s == "penetration").mean())).reset_index()
P("\nBy promo type:\n"); P(dt.to_markdown(index=False, floatfmt=".3f"))

P("\n### Do volume-driven promotions show more pull-forward than penetration-driven ones?\n")
cmp = M.compare_pullforward(ok, "driver", ["volume", "penetration"])
P(cmp.to_markdown(index=False, floatfmt=".3f"))
d = cmp.attrs.get("diff", {})
if d:
    P(f"\n- Difference (volume minus penetration) in pull-forward ratio: **{d['diff']:+.1%}** "
      f"(95% bootstrap CI {d['lo']:+.1%} to {d['hi']:+.1%}, bootstrap p = {d['p_boot']:.3f}).")
    # also Welch on per-event post incremental as share of in-week
    a = ok[(ok.driver == "volume") & (ok.n_post_clean == 4)]; b = ok[(ok.driver == "penetration") & (ok.n_post_clean == 4)]
    ra = (a.post_incr / a.incr_A).clip(-3, 3); rb = (b.post_incr / b.incr_A).clip(-3, 3)
    t, pv = stats.ttest_ind(ra, rb, equal_var=False)
    P(f"- Welch t-test on per-event post/in-week ratio (clipped to +/-3): volume mean {ra.mean():+.3f} vs penetration {rb.mean():+.3f}, t = {t:.2f}, p = {pv:.4f}.")
    summary["volume_vs_penetration"] = {**d, "welch_t": float(t), "welch_p": float(pv), "n_volume": int(len(a)), "n_penetration": int(len(b))}
# by category within driver
cmp_cat = []
for c in sorted(ok.category.unique()):
    r = M.compare_pullforward(ok[ok.category == c], "driver", ["volume", "penetration"], n_boot=500)
    r["category"] = c; cmp_cat.append(r)
cmp_cat = pd.concat(cmp_cat)[["category", "driver", "n_events", "pullforward_ratio", "lo", "hi"]]
P("\nBy category and driver:\n"); P(cmp_cat.to_markdown(index=False, floatfmt=".3f"))
summary["decomposition"] = {"share_penetration_driven": float((ok.driver == "penetration").mean()),
                            "pen_share_mean": float(ok.pen_share.clip(-1, 2).mean()), "by_category": dc.to_dict(orient="records"),
                            "pullforward_by_driver": cmp.to_dict(orient="records"), "by_category_driver": cmp_cat.to_dict(orient="records")}

fig, ax = plt.subplots(figsize=(6, 5))
sm = ok.sample(min(8000, len(ok)), random_state=0)
ax.scatter(sm.dln_hhs, sm.dln_uph, s=4, alpha=.3, c=np.where(sm.driver == "penetration", "#1f6f8b", "#c0392b"))
ax.axhline(0, color="k", lw=.6); ax.axvline(0, color="k", lw=.6)
lim = [-1, 2.5]; ax.plot(lim, lim, ls="--", color="grey", lw=.8, label="equal contribution")
ax.set_xlim(lim); ax.set_ylim([-1, 1.5])
ax.set_xlabel("households component  ln(hhs / baseline hhs)"); ax.set_ylabel("units-per-household component")
ax.set_title("Decomposition of log lift per event\n(blue = penetration-driven, red = volume-driven)"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase2_decomposition.png", dpi=130); plt.close(fig)

In [ ]:
# ------------------------------------------------------------------ 2d limitations (data-driven bits)
P("\n## 2d. Limitations surfaced by the data\n")
P(f"- {ev.has_controls.mean():.0%} of events have usable control stores; Method B is a cross-check, not the headline.")
P(f"- {(ev.n_post_clean==4).mean():.0%} of events have a clean 4-week post window; pull-forward is measured on those, and they are "
  "events followed by a promotion gap, which may differ from back-to-back promotions.")
P("- Promotions are not randomly assigned; promo frequency correlates +0.29 with seasonal demand (Phase 0), handled by week-of-year features but not provably removed.")
P("- Cannibalisation across the 55 UPCs in a category is NOT measured here (out of scope; would need a category-level model).")

ev.to_parquet(PROC / "events_phase2.parquet", index=False)
json.dump(summary, open(ROOT / "outputs" / "phase2_summary.json", "w"), indent=2, default=float)
OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))